# El Niño 2026 — Notebook 02 (v2): the trigger (H2) and the Kelvin waves
**le-gis.com climate project**

Notebook 01 showed *that* heat was stored (recharge) and *that* ocean and atmosphere coupled (Bjerknes).
This notebook shows *how* the heat moved east, using **Hovmöller diagrams** (longitude across the equatorial Pacific × time).

| Panel | Variable | What to look for | Source |
|---|---|---|---|
| a | 10 m zonal wind anomaly | **Westerly wind bursts (WWBs)** in the west Pacific | **ERA5** (Copernicus CDS), 1° daily |
| b | Outgoing longwave radiation (OLR) anomaly | Convection (negative OLR) leaving Indonesia, moving east | **ERA5** top net thermal radiation |
| c | Sea surface temperature anomaly | Warming spreading east to South America | **NOAA OISST v2.1** via Earth Engine (same data as NOAA's Niño indices) |
| d | 20 °C isotherm depth (D20) anomaly | **Downwelling Kelvin waves**: diagonal bands sloping east | **GLORYS12** + Copernicus analysis (Copernicus Marine) |
| e | Temperature anomaly, depth × longitude | The subsurface warm pool tilting east | GLORYS12 |

Equator band 5°S–5°N (atmosphere) / 2°S–2°N (ocean); 120°E–80°W. Baselines: 1991–2020 (ERA5, OISST); 1993–2020 (GLORYS starts 1993).

Causal chain tested (Harrison & Vecchi 1997; Kessler et al. 1995; McPhaden 1999):
**WWB → downwelling Kelvin wave (~2.5 m/s) → deeper thermocline in the east → warmer upwelled water → SST rise at Niño-1+2 → weaker trades (Bjerknes).**

### One-time setup (≈10 min)
1. **Copernicus Climate Data Store**: register at cds.climate.copernicus.eu → profile page → copy your **Personal Access Token**. Then open the dataset page *ERA5 hourly data on single levels* → **Download** tab → accept the **Terms of use** (requests fail without this).
2. **Copernicus Marine**: register at marine.copernicus.eu (username + password).
3. Earth Engine project ID (same as Notebook 01).

First run downloads the 30-year baselines (≈30–60 min, mostly waiting in the CDS queue). Everything is cached to Drive, so later runs only fetch the newest days.

In [ ]:
!pip -q install "cdsapi>=0.7.4" copernicusmarine earthengine-api netCDF4
import os, time, getpass, warnings
import numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt, matplotlib.dates as mdates
from concurrent.futures import ThreadPoolExecutor
warnings.filterwarnings("ignore")
plt.rcParams.update({"axes.grid": True, "grid.alpha": .25, "font.size": 9})

from google.colab import drive
drive.mount("/content/drive")
CACHE = "/content/drive/MyDrive/le-gis_enso/cache"; os.makedirs(CACHE, exist_ok=True)

START, END = "2025-07-01", None
LON0, LON1 = 120, 280            # 120E .. 80W (0-360)
ZMAX = 400                       # metres
TODAY = pd.Timestamp.today().normalize()
this_year = TODAY.year
years_now = list(range(pd.Timestamp(START).year, this_year + 1))

def clean_time(da):
    da = da.sortby("time")
    _, keep = np.unique(da.time.values, return_index=True)
    return da.isel(time=np.sort(keep))

def daily_anom(da_now, da_clim):
    # Anomaly from a smoothed (31-day) day-of-year climatology.
    doy = da_clim.groupby("time.dayofyear").mean()
    doy = xr.concat([doy.isel(dayofyear=slice(-15, None)), doy, doy.isel(dayofyear=slice(0, 15))], "dayofyear")
    doy = doy.rolling(dayofyear=31, center=True).mean().isel(dayofyear=slice(15, -15))
    return da_now.groupby("time.dayofyear") - doy

In [ ]:
# --- Credentials (stored only in this Colab session) ---
if not os.path.exists(os.path.expanduser("~/.cdsapirc")):
    token = getpass.getpass("CDS Personal Access Token: ")
    with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
        f.write(f"url: https://cds.climate.copernicus.eu/api\nkey: {token}\n")

import copernicusmarine as cm
cm.login(username=input("Copernicus Marine username: "),
         password=getpass.getpass("Copernicus Marine password: "), force_overwrite=True)

import ee
PROJECT = "YOUR-EE-PROJECT"   # <-- change
ee.Authenticate(); ee.Initialize(project=PROJECT)

## a–b. ERA5: 10 m zonal wind and OLR
Requests a 1° grid, 4 times per day (00/06/12/18 UTC), equatorial strip only → small files.
OLR = −(top net thermal radiation) / 3600 s, in W/m². Past years are downloaded 4 at a time and cached.

In [ ]:
import cdsapi
cds = cdsapi.Client(quiet=True)
GRID = [1.0, 1.0]          # set to None if the CDS rejects 'grid' (then 0.25 deg, ~6x larger files)
ERA5_VARS = {"u": "10m_u_component_of_wind", "olr": "top_net_thermal_radiation"}

def process_era5(path, short):
    ds = xr.open_dataset(path)
    ren = {k: v for k, v in {"valid_time": "time", "latitude": "lat", "longitude": "lon"}.items() if k in ds.variables}
    ds = ds.rename(ren)
    if "expver" in ds.dims:
        ds = ds.mean("expver")
    name = [v for v in ds.data_vars if v in ("u10", "ttr")][0]
    da = ds[name]
    da = da.assign_coords(lon=da.lon % 360).sortby("lon").drop_duplicates("lon").sortby("lat")
    da = da.sel(lon=slice(LON0, LON1))
    da = da.weighted(np.cos(np.deg2rad(da.lat))).mean("lat")
    da = da.resample(time="1D").mean()
    if name == "ttr":
        da = -da / 3600.0                       # J/m2 per hour -> W/m2, positive upward
    out = da.rename(short).load()
    ds.close()
    return out

import threading
NC_LOCK = threading.Lock()       # netCDF/HDF5 is not thread-safe: one file operation at a time

def era5_chunk(short, year, months, days, cache=True):
    tag = f"{short}_{year}_m{months[0]:02d}-{months[-1]:02d}_d{days[-1]:02d}"
    f = f"{CACHE}/era5_{tag}.nc"
    if cache and os.path.exists(f):
        try:
            with NC_LOCK:
                return xr.open_dataarray(f).load()
        except Exception:
            print(f"  {tag}: cached file unreadable (interrupted write?) -> re-downloading")
            os.remove(f)
    raw = f"/content/_era5_{tag}.nc"
    req = {"product_type": ["reanalysis"], "variable": [ERA5_VARS[short]],
           "year": [str(year)], "month": [f"{m:02d}" for m in months], "day": [f"{d:02d}" for d in days],
           "time": ["00:00", "06:00", "12:00", "18:00"], "area": [5, LON0, -5, LON1],
           "data_format": "netcdf", "download_format": "unarchived"}
    if GRID:
        req["grid"] = GRID
    for k in range(3):
        try:
            cds.retrieve("reanalysis-era5-single-levels", req).download(raw); break
        except Exception as e:
            if k == 2: raise
            print(f"  {tag}: retry {k+1} ({str(e)[:80]})"); time.sleep(30)
    with NC_LOCK:
        da = process_era5(raw, short); os.remove(raw)
        if cache:
            da.to_netcdf(f + ".tmp"); os.replace(f + ".tmp", f)   # atomic: no half-written cache
    return da

def era5_series(short, years, workers=4):
    cutoff = TODAY - pd.Timedelta(days=6)                    # ERA5T latency ~5 days
    full = [y for y in years if y < cutoff.year]
    with ThreadPoolExecutor(workers) as ex:                  # past years in parallel
        parts = list(ex.map(lambda y: era5_chunk(short, y, list(range(1, 13)), list(range(1, 32))), full))
    if cutoff.year in years:
        if cutoff.month > 1:
            parts.append(era5_chunk(short, cutoff.year, list(range(1, cutoff.month)), list(range(1, 32))))
        parts.append(era5_chunk(short, cutoff.year, [cutoff.month], list(range(1, cutoff.day + 1)), cache=False))
    print(f"{short}: {len(full)} full years + current year loaded")
    return clean_time(xr.concat(parts, "time"))

print("ERA5 wind ...")
u_clim = era5_series("u", list(range(1991, 2021)))
u_now = era5_series("u", years_now).sel(time=slice(START, END))
ua = daily_anom(u_now, u_clim)
print("ERA5 OLR ...")
olr_clim = era5_series("olr", list(range(1991, 2021)))
olr_now = era5_series("olr", years_now).sel(time=slice(START, END))
olr_a = daily_anom(olr_now, olr_clim)
print("Wind/OLR anomaly:", str(ua.time.values[0])[:10], "->", str(ua.time.values[-1])[:10])

In [ ]:
def detect_wwb(ua, thresh=5.0, min_width=10.0, min_days=2, lon_range=(130, 210)):
    # WWB day: anomaly >= thresh (m/s) over a contiguous span >= min_width degrees inside lon_range.
    # Consecutive WWB days are grouped into events (after Harrison & Vecchi 1997).
    sub = ua.sel(lon=slice(*lon_range))
    dlon = float(np.diff(sub.lon.values).mean())
    rows = []
    for t in range(sub.sizes["time"]):
        v = sub.isel(time=t).values
        best, run, start, span = 0, 0, 0, None
        for i, m in enumerate(np.append(v >= thresh, False)):
            if m:
                if run == 0: start = i
                run += 1
            else:
                if run > best: best, span = run, (start, i - 1)
                run = 0
        if best * dlon >= min_width:
            i0, i1 = span
            rows.append((pd.Timestamp(sub.time.values[t]), float(v[i0:i1+1].max()),
                         float(sub.lon.values[i0:i1+1].mean())))
    if not rows:
        return pd.DataFrame(columns=["start", "end", "days", "peak_ms", "center_lon"])
    d = pd.DataFrame(rows, columns=["date", "peak", "lon"])
    d["event"] = (d.date.diff().dt.days.fillna(99) > 2).cumsum()
    ev = d.groupby("event").agg(start=("date", "min"), end=("date", "max"),
                                peak_ms=("peak", "max"), center_lon=("lon", "mean"))
    ev["days"] = (ev.end - ev.start).dt.days + 1
    ev[["peak_ms", "center_lon"]] = ev[["peak_ms", "center_lon"]].round(1)
    return ev[ev.days >= min_days][["start", "end", "days", "peak_ms", "center_lon"]].reset_index(drop=True)

wwb = detect_wwb(ua)
wwb.to_csv("wwb_events_2025_2026.csv", index=False)
wwb

## c. SST anomaly — NOAA OISST v2.1 from Earth Engine
Daily profiles along the equator (2°S–2°N), averaged into 1° longitude bins on the server; one small request per month.

In [ ]:
oisst = ee.ImageCollection("NOAA/CDR/OISST/V2_1").select("anom")
STRIP = ee.Geometry.MultiPolygon(
    [[[[120, -2], [180, -2], [180, 2], [120, 2], [120, -2]]],
     [[[-180, -2], [-80, -2], [-80, 2], [-180, 2], [-180, -2]]]], None, False)
LONBIN = ee.Image.pixelLonLat().select("longitude").floor().int().rename("lonbin")

def day_profile(img):
    stats = img.multiply(0.01).rename("sst").addBands(LONBIN).reduceRegion(
        reducer=ee.Reducer.mean().group(groupField=1, groupName="lonbin"),
        geometry=STRIP, scale=27830, maxPixels=1e9)
    return ee.Feature(None, {"date": img.date().format("YYYY-MM-dd"), "groups": stats.get("groups")})

def profiles_to_da(rows):
    df = pd.DataFrame(rows).T
    df.index = pd.to_datetime(df.index)
    df = df.sort_index().sort_index(axis=1)
    da = xr.DataArray(df.values.astype(float), dims=["time", "lon"],
                      coords={"time": df.index, "lon": df.columns.astype(float)}, name="sst_anom")
    return da.sel(lon=slice(LON0, LON1))

rows = {}
for m in pd.date_range(START, TODAY, freq="MS"):
    nxt = m + pd.offsets.MonthBegin(1)
    feats = ee.FeatureCollection(oisst.filterDate(str(m.date()), str(nxt.date())).map(day_profile)).getInfo()["features"]
    for f in feats:
        p = f["properties"]
        rows[p["date"]] = {(g["lonbin"] + 0.5) % 360: g["mean"] for g in (p["groups"] or [])}
sst_a = profiles_to_da(rows)
print("SST anomaly:", str(sst_a.time.values[0])[:10], "->", str(sst_a.time.values[-1])[:10])

## d–e. Subsurface ocean — GLORYS12 (Copernicus Marine)
Monthly 1/12° potential temperature, 0–400 m, 2°S–2°N → averaged to 1° longitude.
Each month is taken from the first product that covers it: **reanalysis → interim → real-time analysis**. The switch to the real-time analysis is marked on the figure.

In [ ]:
GLORYS = [("cmems_mod_glo_phy_my_0.083deg_P1M-m", "reanalysis"),
          ("cmems_mod_glo_phy_myint_0.083deg_P1M-m", "interim"),
          ("cmems_mod_glo_phy-thetao_anfc_0.083deg_P1M-m", "analysis")]

def month_start(t):
    return pd.to_datetime(np.atleast_1d(t)).to_period("M").to_timestamp()

def cm_open(ds_id, lon_a, lon_b, t0=None, t1=None):
    kw = dict(dataset_id=ds_id, variables=["thetao"], minimum_longitude=lon_a, maximum_longitude=lon_b,
              minimum_latitude=-2, maximum_latitude=2, minimum_depth=0, maximum_depth=ZMAX)
    if t0 is not None:
        kw.update(start_datetime=str(t0.date()), end_datetime=str(t1.date()))
    return cm.open_dataset(**kw)["thetao"]

COVER = {}
for ds_id, label in GLORYS:
    try:
        t = month_start(cm_open(ds_id, 150, 151).time.values)
        COVER[ds_id] = (t.min(), t.max()); print(f"{label:10s} {t.min():%Y-%m} -> {t.max():%Y-%m}  {ds_id}")
    except Exception as e:
        print(f"{label:10s} unavailable ({str(e)[:70]}) - check the ID in the Copernicus Marine catalogue")

def cm_postprocess(parts):
    # parts: lazy DataArrays (time, depth, latitude, longitude) for the two sides of the dateline.
    out = []
    for da in parts:
        out.append(da.weighted(np.cos(np.deg2rad(da.latitude))).mean("latitude"))
    da = xr.concat(out, "longitude")
    da = da.assign_coords(longitude=da.longitude % 360).sortby("longitude").drop_duplicates("longitude")
    step = float(np.median(np.diff(da.longitude.values)))
    n = max(1, int(round(1.0 / step)))
    da = da.coarsen(longitude=n, boundary="trim").mean().load()
    da = da.rename({"longitude": "lon", "depth": "level"})
    da["time"] = month_start(da.time.values)
    return da

def glorys_year(y):
    f = f"{CACHE}/glorys_eq_{y}.nc"
    if os.path.exists(f) and y <= this_year - 2:
        return xr.open_dataarray(f).load()
    want = pd.date_range(f"{y}-01-01", f"{y}-12-01", freq="MS")
    want = want[want <= TODAY]
    pieces = []
    for ds_id, label in GLORYS:
        if ds_id not in COVER or len(want) == 0:
            continue
        a, b = COVER[ds_id]
        sel = want[(want >= a) & (want <= b)]
        if len(sel) == 0:
            continue
        t1 = sel[-1] + pd.offsets.MonthEnd(0)
        for k in range(3):
            try:
                da = cm_postprocess([cm_open(ds_id, 120, 180, sel[0], t1), cm_open(ds_id, -180, -80, sel[0], t1)])
                break
            except Exception as e:
                if k == 2: raise
                print(f"  {y} {label}: retry {k+1} ({str(e)[:60]})"); time.sleep(20)
        pieces.append(da.assign_coords(source=("time", [label] * da.sizes["time"])))
        want = want.difference(sel)
    da = clean_time(xr.concat(pieces, "time")).rename("temp")
    if y <= this_year - 2:
        da.to_netcdf(f)
    print(f"  GLORYS {y}: {da.sizes['time']} months ({', '.join(sorted(set(da.source.values)))})")
    return da

def d20(temp):
    # Depth of the 20 deg C isotherm by linear interpolation; temp dims (..., level).
    T = temp.transpose(..., "level").values
    z = temp.level.values
    below = T < 20
    k = np.clip(np.argmax(below, axis=-1), 1, len(z) - 1)
    t1 = np.take_along_axis(T, (k - 1)[..., None], -1)[..., 0]
    t2 = np.take_along_axis(T, k[..., None], -1)[..., 0]
    out = z[k - 1] + (20 - t1) * (z[k] - z[k - 1]) / (t2 - t1)
    out[~below.any(axis=-1)] = np.nan
    dims = [d for d in temp.transpose(..., "level").dims if d != "level"]
    return xr.DataArray(out, dims=dims, coords={d: temp[d] for d in dims}, name="d20")

print("GLORYS climatology 1993-2020 (first run ~15-30 min, then cached) ...")
clim_T = clean_time(xr.concat([glorys_year(y) for y in range(1993, 2021)], "time"))
T_clim_m = clim_T.groupby("time.month").mean()
D20_clim_m = d20(clim_T).groupby("time.month").mean()

T_now = clean_time(xr.concat([glorys_year(y) for y in years_now], "time")).sel(time=slice(START, END))
T_anom = T_now.groupby("time.month") - T_clim_m
D20_anom = d20(T_now).groupby("time.month") - D20_clim_m
src = pd.Series(T_now.source.values, index=pd.to_datetime(T_now.time.values))
JOIN = src[src == "analysis"].index.min()       # first month from the real-time analysis (or NaT)
print("GLORYS to", f"{T_now.time.values[-1]}"[:7], "| sources:", src.groupby(src).size().to_dict())

## The four-panel Hovmöller — the core figure of the story

In [ ]:
def lon_labels(ax):
    ticks = [120, 150, 180, 210, 240, 270]
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{t}°E" if t < 180 else ("180°" if t == 180 else f"{360-t}°W") for t in ticks])

def hov(ax, da, levels, cmap, title, unit):
    cf = ax.contourf(da.lon, da.time, da.transpose("time", "lon"), levels=levels, cmap=cmap, extend="both")
    lon_labels(ax); ax.set_title(title, loc="left", fontweight="bold")
    ax.yaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    plt.colorbar(cf, ax=ax, orientation="horizontal", pad=0.06, fraction=0.05, label=unit)

t_min = min(pd.Timestamp(x.time.values[0]) for x in (ua, sst_a, D20_anom))
t_max = max(pd.Timestamp(x.time.values[-1]) for x in (ua, sst_a, D20_anom))
fig, ax = plt.subplots(1, 4, figsize=(18, 9), sharey=True)
hov(ax[0], ua.rolling(time=5, center=True).mean(), np.arange(-8, 8.1, 1), "RdBu_r",
    "a  Zonal wind anomaly (ERA5)", "m/s (red = westerly)")
for _, e in wwb.iterrows():
    ax[0].plot([e.center_lon], [e.start + (e.end - e.start) / 2], marker=">", c="k", ms=6)
hov(ax[1], olr_a.rolling(time=5, center=True).mean(), np.arange(-50, 51, 5), "BrBG_r",
    "b  OLR anomaly (ERA5)", "W/m² (green = more convection)")
hov(ax[2], sst_a.rolling(time=5, center=True).mean(), np.arange(-4, 4.1, .5), "RdBu_r",
    "c  SST anomaly (OISST)", "°C")
hov(ax[3], D20_anom, np.arange(-60, 61, 10), "PuOr_r",
    "d  D20 anomaly (GLORYS)", "m (orange = deeper, warmer)")
if pd.notna(JOIN):
    ax[3].axhline(JOIN, c="k", ls="--", lw=1)
    ax[3].text(122, JOIN, " real-time analysis below", va="bottom", fontsize=7)
ax[0].set_ylim(t_max, t_min)                 # time runs downward
fig.suptitle("Equatorial Pacific, 2025–2026: wind bursts → Kelvin waves → surface warming",
             fontsize=13, fontweight="bold", x=0.01, ha="left")
plt.tight_layout(); plt.savefig("hovmoller_4panel.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# e. Depth x longitude section: latest month vs one year earlier
latest = pd.Timestamp(T_anom.time.values[-1])
before = pd.Timestamp(T_anom.sel(time=latest - pd.DateOffset(years=1), method="nearest").time.values)
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
for a, t in zip(ax, [before, latest]):
    sec = T_anom.sel(time=t)
    cf = a.contourf(sec.lon, sec.level, sec.transpose("level", "lon"), levels=np.arange(-8, 8.1, 1),
                    cmap="RdBu_r", extend="both")
    a.contour(T_now.lon, T_now.level, T_now.sel(time=t).transpose("level", "lon"),
              levels=[20], colors="k", linewidths=1.5)
    a.invert_yaxis(); lon_labels(a); a.set_title(t.strftime("%B %Y"), loc="left", fontweight="bold")
ax[0].set_ylabel("Depth (m)")
plt.colorbar(cf, ax=ax, label="Temperature anomaly (°C); black line = 20 °C isotherm")
plt.savefig("subsurface_section.png", dpi=200, bbox_inches="tight"); plt.show()

## Testing H2 with numbers
1. **Kelvin wave speed**: track the D20 anomaly maximum eastward (first-mode Kelvin wave ≈ 2.4–2.8 m/s).
2. **Lag WWB → Niño-1+2**: do coastal SST jumps follow westerly forcing by ~2 months?

In [ ]:
peak_lon = D20_anom.sel(lon=slice(180, 270)).idxmax("lon").to_series()
print("Longitude of deepest thermocline anomaly by month:\n", peak_lon.round(0).tail(12))
seg = peak_lon.tail(6)
dx = np.diff(seg.values) * 111e3
dt = np.diff(seg.index.values).astype("timedelta64[s]").astype(float)
spd = dx / dt
print(f"Mean eastward speed of the D20 maximum, last 6 months: {np.nanmean(spd[spd > 0]):.2f} m/s "
      "(monthly data -> rough; first-mode Kelvin wave ~2.5 m/s)")

wind_index = ua.sel(lon=slice(140, 200)).mean("lon").to_series()
n12 = sst_a.sel(lon=slice(270, 280)).mean("lon").to_series()          # 90W-80W on the equator
df = pd.concat([wind_index.rename("u"), n12.rename("n12")], axis=1).dropna()
lags = range(0, 121, 5)
r = [df["n12"].diff(30).corr(df["u"].rolling(30).mean().shift(L)) for L in lags]
best = lags[int(np.nanargmax(r))]
plt.figure(figsize=(9, 3.5))
plt.bar(list(lags), r, width=4, color=["crimson" if L == best else "steelblue" for L in lags])
plt.title(f"30-day westerly forcing (140°E–160°W) leading 30-day Niño-1+2 warming — best lag {best} days")
plt.xlabel("lag (days)"); plt.ylabel("correlation"); plt.show()

In [ ]:
# Save compact data for the web (interactive Hovmöller later)
OUT = "/content/drive/MyDrive/le-gis_enso/notebook02"; os.makedirs(OUT, exist_ok=True)
xr.Dataset({"u_anom": ua, "olr_anom": olr_a}).to_netcdf(f"{OUT}/hov_atmos_daily_era5.nc")
sst_a.to_netcdf(f"{OUT}/hov_sst_daily_oisst.nc")
xr.Dataset({"d20_anom": D20_anom, "temp_anom": T_anom.drop_vars("source", errors="ignore")}
           ).to_netcdf(f"{OUT}/glorys_monthly_anom.nc")
for f in ["hovmoller_4panel.png", "subsurface_section.png", "wwb_events_2025_2026.csv"]:
    os.system(f"cp {f} {OUT}/")
print("Saved to", OUT)

## Reading the result (fill in after running)
- **WWBs**: dates and longitudes from the table → do they precede the D20 bands?
- **Kelvin waves**: number of downwelling pulses in 2026 and their speed.
- **Convection shift**: when did negative OLR leave the Maritime Continent? That is when Indonesia's dry season began to fail (link to Notebook 01 impact map).
- **Verdict on H2**: supported / partly supported.

**Next — Notebook 03**: synthesis — what made 2026 historic (recharge + WWBs + warm baseline + positive IOD), comparison with 1997 and 2015, web export.

**Data**: ERA5 (Hersbach et al. 2020, *QJRMS*; Copernicus C3S); GLORYS12V1 (Lellouche et al. 2021, *Front. Earth Sci.*; Copernicus Marine);
NOAA OISST v2.1 (Huang et al. 2021, *J. Climate*). **Science**: Harrison & Vecchi 1997 *J. Climate*; Kessler, McPhaden & Weickmann 1995 *JGR*;
McPhaden 1999 *Science*; Meinen & McPhaden 2000 *J. Climate*.